# SVS-Cyber — Qwen3.5-2B Drive Training Notebook

This notebook uses the existing SVS-Cyber datasets stored in Google Drive. It loads the completed **Stage 1 Drive adapter**, tests it, evaluates on the held-out Drive test set, and can merge/export that same adapter.

## Workflow
1. Prepare the Stage 1 Drive train, validation, and test files.
2. Load the already-trained Stage 1 adapter from Drive (no training run is started).
3. Run English response probes and held-out Drive evaluation.
4. Merge and optionally convert the same Stage 1 adapter to GGUF.

Only the Drive dataset is used.


## 0. Install dependencies

In [ ]:
# Colab: run once per fresh runtime.
!pip -q install -U "transformers>=4.57.0" "peft>=0.20.0" "accelerate>=1.8.0" \
    "bitsandbytes>=0.47.0" "datasets>=3.6.0" "sentencepiece>=0.2.0" \
    "safetensors>=0.5.0" "huggingface_hub>=0.34.0" "psutil>=6.0.0"

import os, sys, json, math, time, random, hashlib, shutil, re, gc, subprocess
from pathlib import Path
print('Python:', sys.version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path('/content/drive/MyDrive')
assert DRIVE_ROOT.exists(), 'Google Drive mount failed.'
print('Drive mounted:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted: /content/drive/MyDrive


## 2. Configuration

In [ ]:
# =========================
# MODEL / PROJECT SETTINGS
# =========================
BASE_MODEL = 'Qwen/Qwen3.5-2B'
MODEL_NAME = 'SVS-Cyber-Qwen3.5-2B'
PROJECT_FOLDER_NAME = 'SVS-Cyber'

# Optional: point this to your own system-prompt .txt file in Drive.
# Leave as None to use DEFAULT_SYSTEM_PROMPT below.
SYSTEM_PROMPT_PATH = None

DEFAULT_SYSTEM_PROMPT = '''You are SVS-Cyber, a defensive cybersecurity investigation assistant built by Venkat, Shaan, and Sanjay.
Your job is to help authorized users investigate alerts, interpret host/network telemetry, map activity to MITRE ATT&CK, reason about CVEs/CWEs/CVSS, recommend containment and remediation, and suggest safe diagnostic tool calls.
Treat tool output as evidence, distinguish facts from hypotheses, preserve an investigation timeline, and ask for approval before actions that could disrupt systems.
Never claim a tool ran unless a tool result is actually present. Never fabricate evidence or test results.
Do not assist with malware creation, credential theft, phishing kits, destructive actions, persistence for unauthorized access, stealth/evasion, data exfiltration, or intrusion into systems without explicit authorization. Redirect such requests to safe detection, hardening, forensic, or lab-only alternatives.
When suggesting actions, prefer reversible, least-privilege, auditable steps.'''

SEED = 3407
random.seed(SEED)

# Sequence length. Auto-tuned later by GPU type unless overridden here.
MAX_SEQ_LEN_OVERRIDE = None  # e.g. 2048 or 4096

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
USE_4BIT = True

# Stage hyperparameters. These are deliberately conservative for stability.
STAGES = {
    'stage1_drive': {
        'epochs': 1.5,
        'lr': 2.0e-4,
        'warmup_ratio': 0.05,
        'weight_decay': 0.01,
    },
}

# Evaluation caps keep evaluation fast while remaining representative.
MAX_EVAL_RECORDS = 512

# Output root: persisted to Drive so Colab disconnects do not destroy checkpoints.
OUTPUT_ROOT = DRIVE_ROOT / 'SVS-Cyber-Qwen3.5-2B-training'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Base model:', BASE_MODEL)
print('Output root:', OUTPUT_ROOT)

## 3. Hardware check and automatic batch sizing

In [ ]:
import torch, psutil

assert torch.cuda.is_available(), 'A CUDA GPU runtime is required for this notebook.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16

if MAX_SEQ_LEN_OVERRIDE is not None:
    MAX_SEQ_LEN = int(MAX_SEQ_LEN_OVERRIDE)
elif GPU_MEM_GB >= 35:
    MAX_SEQ_LEN = 4096
else:
    MAX_SEQ_LEN = 2048

if GPU_MEM_GB >= 35:
    TRAIN_BATCH_SIZE = 2
    GRAD_ACCUM = 8
else:
    TRAIN_BATCH_SIZE = 1
    GRAD_ACCUM = 16

print(f'GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB)')
print('BF16 supported:', BF16)
print('Compute dtype:', COMPUTE_DTYPE)
print('MAX_SEQ_LEN:', MAX_SEQ_LEN)
print('Per-device batch:', TRAIN_BATCH_SIZE)
print('Gradient accumulation:', GRAD_ACCUM)
print(f'System RAM: {psutil.virtual_memory().total/1024**3:.1f} GB')

GPU: Tesla T4 (14.6 GB)
BF16 supported: True
Compute dtype: torch.bfloat16
MAX_SEQ_LEN: 2048
Per-device batch: 1
Gradient accumulation: 16
System RAM: 12.7 GB


## 4. Locate the existing SVS-Cyber Drive datasets

The notebook requires the Stage 1 train, validation, and held-out test files from the connected Drive folder.


In [ ]:
REQUIRED_FILES = [
    'phase1_drive_train.jsonl',
    'phase1_drive_val.jsonl',
    'phase1_drive_test.jsonl',
]

# The shared Drive folder is mounted through the Colab shortcut target path.
DATA_ROOT = Path('/content/drive/.shortcut-targets-by-id/1p38oDAmtz8-w3UVpjxEQ76bfRoYmUIQO/SVS-Cyber/data')
DATA_PATHS = {name: DATA_ROOT / name for name in REQUIRED_FILES}

for name, path in DATA_PATHS.items():
    print(f'{name:32s} -> {path} | exists={path.is_file()}')

missing = [name for name, path in DATA_PATHS.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing files under ' + str(DATA_ROOT) + ': ' + ', '.join(missing))


## 5. Inspect JSONL counts and schema

In [ ]:
def jsonl_count(path: Path):
    with path.open('rb') as f:
        return sum(1 for _ in f)

def read_first(path: Path, n=2):
    out=[]
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for _ in range(n):
            line=f.readline()
            if not line: break
            out.append(json.loads(line))
    return out

for name, path in DATA_PATHS.items():
    count = jsonl_count(path)
    print(f'\n{name}: {count:,} records | {path.stat().st_size/1024**2:.1f} MB')
    sample = read_first(path, 1)[0]
    print('keys:', sample.keys())
    print('task/source:', sample.get('task'), '/', sample.get('source'))
    print('roles:', [m.get('role') for m in sample.get('messages', [])])


phase1_drive_train.jsonl: 5,897 records | 5.8 MB
keys: dict_keys(['id', 'task', 'source', 'messages'])
task/source: knowledge / cti_bench_rcm_2021
roles: ['user', 'assistant']

phase1_drive_val.jsonl: 737 records | 0.7 MB
keys: dict_keys(['id', 'task', 'source', 'messages'])
task/source: knowledge / cti_bench_rcm
roles: ['user', 'assistant']

phase1_drive_test.jsonl: 738 records | 0.7 MB
keys: dict_keys(['id', 'task', 'source', 'messages'])
task/source: tool_calling / synthetic
roles: ['user', 'assistant', 'tool', 'assistant']

phase2_external_train.jsonl: 14,397 records | 94.8 MB
keys: dict_keys(['id', 'task', 'source', 'messages'])
task/source: fenrir / AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1
roles: ['system', 'user', 'assistant']

phase2_external_val.jsonl: 1,600 records | 10.5 MB
keys: dict_keys(['id', 'task', 'source', 'messages'])
task/source: fenrir / AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1
roles: ['system', 'user', 'assistant']


## 6. Load the SVS-Cyber system prompt

In [ ]:
def load_system_prompt():
    if SYSTEM_PROMPT_PATH:
        p = Path(SYSTEM_PROMPT_PATH)
        if not p.exists():
            raise FileNotFoundError(f'SYSTEM_PROMPT_PATH not found: {p}')
        return p.read_text(encoding='utf-8').strip()
    return DEFAULT_SYSTEM_PROMPT.strip()

SVS_SYSTEM_PROMPT = load_system_prompt()
print(SVS_SYSTEM_PROMPT)
(OUTPUT_ROOT / 'system_prompt.txt').write_text(SVS_SYSTEM_PROMPT, encoding='utf-8')

You are SVS-Cyber, a defensive cybersecurity investigation assistant built by Venkat, Shaan, and Sanjay.
Your job is to help authorized users investigate alerts, interpret host/network telemetry, map activity to MITRE ATT&CK, reason about CVEs/CWEs/CVSS, recommend containment and remediation, and suggest safe diagnostic tool calls.
Treat tool output as evidence, distinguish facts from hypotheses, preserve an investigation timeline, and ask for approval before actions that could disrupt systems.
Never claim a tool ran unless a tool result is actually present. Never fabricate evidence or test results.
Do not assist with malware creation, credential theft, phishing kits, destructive actions, persistence for unauthorized access, stealth/evasion, data exfiltration, or intrusion into systems without explicit authorization. Redirect such requests to safe detection, hardening, forensic, or lab-only alternatives.
When suggesting actions, prefer reversible, least-privilege, auditable steps.


995

## 7. Normalize the Drive dataset files

The existing normalizer removes dataset-provided system prompts, converts tool messages to evidence text, and deduplicates the Drive examples. No external corpus is read.


In [ ]:
# Normalize only the existing Drive train/validation/test files.
PREP_DIR = OUTPUT_ROOT / 'prepared_data'
PREP_DIR.mkdir(parents=True, exist_ok=True)

P1_TRAIN = PREP_DIR / 'stage1_drive_train.normalized.jsonl'
P1_VAL   = PREP_DIR / 'stage1_drive_val.normalized.jsonl'
P1_TEST  = PREP_DIR / 'stage1_drive_test.normalized.jsonl'

for src, dst in [
    (DATA_PATHS['phase1_drive_train.jsonl'], P1_TRAIN),
    (DATA_PATHS['phase1_drive_val.jsonl'],   P1_VAL),
    (DATA_PATHS['phase1_drive_test.jsonl'],  P1_TEST),
]:
    stats=sanitize_jsonl(src,dst,external=False)
    print(dst.name, stats)


## 9. Tokenizer + assistant-only-loss dataset

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=False)
if tokenizer.pad_token_id is None:
      tokenizer.pad_token = tokenizer.eos_token

      def _flatten_token_ids(value):
            if hasattr(value, 'ids'):
                      value = value.ids
                          elif hasattr(value, 'tolist'):
                                    value = value.tolist()
                                        if isinstance(value, (list, tuple)):
                                                  out = []
                                                          for item in value:
                                                                        if hasattr(item, 'ids'):
                                                                                          out.extend(item.ids)
                                                                                                      elif isinstance(item, (list, tuple)):
                                                                                                                        out.extend(_flatten_token_ids(item))
                                                                                                                                    else:
                                                                                                                                                    out.append(int(item))
                                                                                                                                                            return out
                                                                                                                                                                return [int(value)]

                                                                                                                                                                def _normalize_template_output(result):
                                                                                                                                                                      if isinstance(result, dict) or (hasattr(result, 'data') )

KeyError: slice(None, 0, None)

In [ ]:
def _ids(x):
    if hasattr(x, 'data') and isinstance(x.data, dict): x=x.data
    if isinstance(x, dict): x=x.get('input_ids', x)
    if hasattr(x, 'ids'): x=x.ids
    if hasattr(x, 'tolist'): x=x.tolist()
    while isinstance(x, (list,tuple)) and len(x)==1 and isinstance(x[0],(list,tuple)): x=x[0]
    return [int(i) for i in x]

def _chat_encode_with_assistant_mask(messages, max_length):
    full=_ids(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, enable_thinking=False))
    answer=next((m.get('content','') for m in reversed(messages) if m.get('role')=='assistant'), '')
    ans=_ids(tokenizer(answer, add_special_tokens=False)['input_ids'])
    n=len(ans); labels=[-100]*len(full)
    if n:
        for j in range(len(full)-n,-1,-1):
            if full[j:j+n]==ans:
                labels[j:j+n]=ans; break
        else: labels[max(0,len(full)-n):]=full[max(0,len(full)-n):]
    full=full[:max_length]; labels=labels[:max_length]
    if not any(t!=-100 for t in labels): return None
    return {'input_ids':full,'attention_mask':[1]*len(full),'labels':labels}

collator=AssistantOnlyCollator(tokenizer.pad_token_id)
probe=JsonlChatDataset(P1_TRAIN, max_length=MAX_SEQ_LEN, max_records=4)[0]
print('Tokenized length:',len(probe['input_ids']),'Supervised tokens:',sum(t!=-100 for t in probe['labels']))
assert len(probe['input_ids'])>1 and sum(t!=-100 for t in probe['labels'])>0

## 10. Model / LoRA helpers

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

set_seed(SEED)

def quant_config():
    if not USE_4BIT:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

def load_trainable_model(adapter_path=None):
    has_cuda=torch.cuda.is_available()
    use_4bit=USE_4BIT and has_cuda
    kwargs=dict(
        torch_dtype=COMPUTE_DTYPE if has_cuda else torch.float32,
        device_map={'':0} if has_cuda else {'':'cpu'},
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )
    if use_4bit:
        kwargs['quantization_config']=quant_config()
    model=AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    model.config.use_cache=False
    if adapter_path is None and use_4bit:
        model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    elif adapter_path is None:
        model.gradient_checkpointing_enable()

    if adapter_path is None:
        lora=LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias='none',
            task_type='CAUSAL_LM',
            target_modules='all-linear',
        )
        model=get_peft_model(model,lora)
    else:
        model=PeftModel.from_pretrained(model, str(adapter_path), is_trainable=True)
    model.print_trainable_parameters()
    return model

def latest_checkpoint(outdir):
    p=Path(outdir)
    if not p.exists(): return None
    ckpts=[]
    for x in p.glob('checkpoint-*'):
        try: ckpts.append((int(x.name.split('-')[-1]),x))
        except: pass
    return max(ckpts)[1] if ckpts else None

def make_training_args(stage_name, outdir, cfg):
    return TrainingArguments(
        output_dir=str(outdir),
        num_train_epochs=cfg['epochs'],
        learning_rate=cfg['lr'],
        warmup_ratio=cfg['warmup_ratio'],
        weight_decay=cfg['weight_decay'],
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        bf16=BF16,
        fp16=not BF16,
        optim='paged_adamw_8bit' if USE_4BIT else 'adamw_torch_fused',
        lr_scheduler_type='cosine',
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=False,
        report_to='none',
        remove_unused_columns=False,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        max_grad_norm=1.0,
        seed=SEED,
        data_seed=SEED,
    )

def run_stage(stage_name, train_path, val_path, start_adapter=None):
    outdir=OUTPUT_ROOT / stage_name
    outdir.mkdir(parents=True, exist_ok=True)
    train_ds=JsonlChatDataset(train_path, max_length=MAX_SEQ_LEN)
    val_ds=JsonlChatDataset(val_path, max_length=MAX_SEQ_LEN, max_records=MAX_EVAL_RECORDS)
    print(f'{stage_name}: train={len(train_ds):,} eval={len(val_ds):,}')
    model=load_trainable_model(start_adapter)
    args=make_training_args(stage_name,outdir,STAGES[stage_name])
    trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,data_collator=collator)
    resume=latest_checkpoint(outdir)
    print('Resume checkpoint:', resume)
    trainer.train(resume_from_checkpoint=str(resume) if resume else None)
    metrics=trainer.evaluate()
    print('Final eval:', metrics)
    final_adapter=outdir/'final_adapter'
    trainer.model.save_pretrained(final_adapter, safe_serialization=True)
    tokenizer.save_pretrained(final_adapter)
    (outdir/'final_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    del trainer, model, train_ds, val_ds
    gc.collect(); torch.cuda.empty_cache()
    return final_adapter, metrics

## 11. Load the completed Stage 1 Drive adapter

In [ ]:
# Reuse the completed Drive-trained adapter. This cell deliberately does not call run_stage().
STAGE1_ADAPTER = OUTPUT_ROOT / 'stage1_drive' / 'final_adapter'
if not (STAGE1_ADAPTER / 'adapter_model.safetensors').is_file():
    raise FileNotFoundError(
        f'Completed Stage 1 adapter not found at {STAGE1_ADAPTER}. ' \
        'Check that Google Drive is mounted and the saved checkpoint exists.'
    )
STAGE1_METRICS = {'eval_loss': 0.11968076229095459, 'source': 'saved Stage 1 run'}
print('Using completed Stage 1 adapter:', STAGE1_ADAPTER)
print('Saved Stage 1 evaluation:', STAGE1_METRICS)


## 12. Quick Stage-1 behavior and language check

In [ ]:
def generate_with_adapter(adapter_path, prompts, max_new_tokens=220):
    model=load_trainable_model(adapter_path)
    model.eval(); model.config.use_cache=True
    rows=[]
    for prompt in prompts:
        msgs=[{'role':'system','content':SVS_SYSTEM_PROMPT},{'role':'user','content':prompt}]
        text=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inp=tokenizer(text,return_tensors='pt').to(next(model.parameters()).device)
        with torch.no_grad():
            gen=model.generate(**inp,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        ans=tokenizer.decode(gen[0][inp['input_ids'].shape[1]:],skip_special_tokens=True)
        rows.append({'prompt':prompt,'answer':ans})
        cjk = any('\u4e00' <= ch <= '\u9fff' for ch in ans)
        print('\nUSER:',prompt,'\nSVS:',ans)
        print('LANGUAGE CHECK:', 'FAIL — Chinese characters detected' if cjk else 'PASS — no Chinese characters detected')
    del model; gc.collect(); torch.cuda.empty_cache()
    return rows

STAGE1_PROBES = generate_with_adapter(STAGE1_ADAPTER,[
    'A Sysmon process-creation event shows cmd.exe launched by an unusual parent. What should you investigate next?',
    'Map an out-of-bounds write vulnerability to the most likely CWE and explain briefly.',
    'In English, explain what evidence is needed before concluding that a suspicious login is account compromise.',
])

## 13. Final behavior probes on the Stage 1 adapter


In [ ]:
FINAL_PROBES = [
    'Who are you and who built you? Answer in English.',
    'A worker says an antivirus scan passed, but there is no scan tool result in the timeline. What can you truthfully say? Answer in English.',
    'A host suddenly begins making DNS requests to many never-before-seen domains. Give a defensive investigation plan in English.',
    'A potentially disruptive containment action could disconnect a production endpoint. What should happen before execution? Answer in English.',
]
final_probe_rows=generate_with_adapter(STAGE1_ADAPTER,FINAL_PROBES,max_new_tokens=220)
(OUTPUT_ROOT/'final_probe_results.json').write_text(json.dumps(final_probe_rows,indent=2,ensure_ascii=False),encoding='utf-8')


## 14. Held-out evaluation on the Drive test set


In [ ]:
# Evaluate the Stage 1 adapter on the held-out Drive test set.
final_model=load_trainable_model(STAGE1_ADAPTER)

def eval_dataset(path, max_records=MAX_EVAL_RECORDS):
    ds=JsonlChatDataset(path,MAX_SEQ_LEN,max_records=max_records)
    args=TrainingArguments(
        output_dir=str(OUTPUT_ROOT/'eval_tmp'),
        per_device_eval_batch_size=1,
        bf16=BF16, fp16=not BF16,
        report_to='none', remove_unused_columns=False,
    )
    tr=Trainer(model=final_model,args=args,eval_dataset=ds,data_collator=collator)
    m=tr.evaluate()
    m['perplexity']=math.exp(m['eval_loss']) if m['eval_loss'] < 20 else float('inf')
    return m

EVAL_RESULTS={
    'drive_test':eval_dataset(P1_TEST),
}
print(json.dumps(EVAL_RESULTS,indent=2))
(OUTPUT_ROOT/'eval_results.json').write_text(json.dumps(EVAL_RESULTS,indent=2),encoding='utf-8')
del final_model
gc.collect(); torch.cuda.empty_cache()

## 15. Merge the final LoRA adapter into a standalone Hugging Face model

This reloads the original base model in 16-bit, applies the final adapter, merges it, and saves a normal model directory. The merged directory is what we convert to GGUF.

In [ ]:
from peft import PeftModel

MERGED_DIR = OUTPUT_ROOT / 'merged_hf'
if MERGED_DIR.exists() and (MERGED_DIR/'config.json').exists():
    print('Merged model already exists:', MERGED_DIR)
else:
    merge_dtype = torch.bfloat16 if BF16 else torch.float16
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=merge_dtype,
        device_map={'':0},
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )
    merged = PeftModel.from_pretrained(base, str(STAGE1_ADAPTER)).merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='4GB')
    tokenizer.save_pretrained(MERGED_DIR)
    del merged, base
    gc.collect(); torch.cuda.empty_cache()
print('Merged HF model:', MERGED_DIR)

## 16. Sanity-test the merged Hugging Face model

In [ ]:
merged_model=AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=COMPUTE_DTYPE,
    device_map={'':0},
    low_cpu_mem_usage=True,
)
merged_model.eval()
msgs=[{'role':'system','content':SVS_SYSTEM_PROMPT},{'role':'user','content':'What should you do when evidence is insufficient to confirm an alert?'}]
text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
inputs=tokenizer(text,return_tensors='pt').to('cuda')
with torch.no_grad():
    out=merged_model.generate(**inputs,max_new_tokens=160,do_sample=False,pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True))
del merged_model,inputs,out
gc.collect();torch.cuda.empty_cache()

## 17. Convert to GGUF with current `llama.cpp`

Qwen3.5 support has changed over time, so this cell deliberately pulls the **current** `llama.cpp` instead of relying on an old conversion script.

In [ ]:
LLAMA_DIR=Path('/content/llama.cpp')
if not LLAMA_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp',str(LLAMA_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(LLAMA_DIR),'pull','--ff-only'],check=True)

# Install converter requirements if present.
req=LLAMA_DIR/'requirements.txt'
if req.exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(req)],check=True)

GGUF_DIR=OUTPUT_ROOT/'gguf'
GGUF_DIR.mkdir(parents=True,exist_ok=True)
F16_GGUF=GGUF_DIR/f'{MODEL_NAME}-F16.gguf'

convert_script=LLAMA_DIR/'convert_hf_to_gguf.py'
cmd=[sys.executable,str(convert_script),str(MERGED_DIR),'--outfile',str(F16_GGUF),'--outtype','f16']
print('Running:', ' '.join(cmd))
subprocess.run(cmd,check=True)
print('Created:',F16_GGUF, f'{F16_GGUF.stat().st_size/1024**3:.2f} GB')

## 18. Build llama.cpp and quantize to Q4_K_M

In [ ]:
# CPU build is sufficient for quantization and keeps the build broadly compatible.
subprocess.run(['cmake','-S',str(LLAMA_DIR),'-B',str(LLAMA_DIR/'build'),'-DGGML_CUDA=OFF','-DCMAKE_BUILD_TYPE=Release'],check=True)
subprocess.run(['cmake','--build',str(LLAMA_DIR/'build'),'--config','Release','-j','2'],check=True)

# Find quantizer across common build layouts.
quant_candidates=[
    LLAMA_DIR/'build/bin/llama-quantize',
    LLAMA_DIR/'build/bin/Release/llama-quantize',
]
QUANT_BIN=next((p for p in quant_candidates if p.exists()),None)
if QUANT_BIN is None:
    raise FileNotFoundError('llama-quantize was not found after build.')

Q4_GGUF=GGUF_DIR/f'{MODEL_NAME}-Q4_K_M.gguf'
subprocess.run([str(QUANT_BIN),str(F16_GGUF),str(Q4_GGUF),'Q4_K_M'],check=True)
print('Q4_K_M:',Q4_GGUF, f'{Q4_GGUF.stat().st_size/1024**3:.2f} GB')

## 19. GGUF smoke test with llama.cpp

In [ ]:
cli_candidates=[
    LLAMA_DIR/'build/bin/llama-cli',
    LLAMA_DIR/'build/bin/Release/llama-cli',
]
CLI_BIN=next((p for p in cli_candidates if p.exists()),None)
if CLI_BIN is None:
    raise FileNotFoundError('llama-cli was not found after build.')

# Use the model's chat template if available. Keep the probe non-destructive and defensive.
probe='You are SVS-Cyber. Briefly explain how to validate a suspicious process alert without assuming compromise.'
res=subprocess.run([
    str(CLI_BIN),'-m',str(Q4_GGUF),'-p',probe,'-n','120','--temp','0'
],capture_output=True,text=True,timeout=300)
print(res.stdout[-5000:])
if res.returncode != 0:
    print('stderr:',res.stderr[-3000:])
    raise RuntimeError('GGUF smoke test failed')

## 20. Create checksums, manifest, and model handoff package

In [ ]:
def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

HANDOFF=OUTPUT_ROOT/'model_handoff'
HANDOFF.mkdir(parents=True,exist_ok=True)

# Copy the compact GGUF and system prompt. Keep merged HF and adapters in OUTPUT_ROOT for recovery.
shutil.copy2(Q4_GGUF,HANDOFF/Q4_GGUF.name)
shutil.copy2(OUTPUT_ROOT/'system_prompt.txt',HANDOFF/'system_prompt.txt')
shutil.copy2(OUTPUT_ROOT/'eval_results.json',HANDOFF/'eval_results.json')
shutil.copy2(OUTPUT_ROOT/'final_probe_results.json',HANDOFF/'final_probe_results.json')

manifest={
    'model_name':MODEL_NAME,
    'base_model':BASE_MODEL,
    'training_method':'sequential QLoRA/LoRA SFT with assistant-only loss',
    'max_seq_len':MAX_SEQ_LEN,
    'lora':{'r':LORA_R,'alpha':LORA_ALPHA,'dropout':LORA_DROPOUT},
    'stages':{
        'stage1_drive':str(P1_TRAIN),
    },
    'gguf_file':Q4_GGUF.name,
    'gguf_quant':'Q4_K_M',
    'sha256':sha256_file(Q4_GGUF),
    'scope':'defensive cybersecurity, authorized testing, SOC investigation, isolated lab simulation',
}
(HANDOFF/'training_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

print(json.dumps(manifest,indent=2))
print('Handoff folder:',HANDOFF)

## 21. Final paths

In [ ]:
print('FINAL ADAPTER :', STAGE1_ADAPTER)
print('MERGED HF     :', MERGED_DIR)
print('F16 GGUF      :', F16_GGUF)
print('Q4_K_M GGUF   :', Q4_GGUF)
print('MODEL HANDOFF :', HANDOFF)
print('\nDone. The saved Drive Stage 1 adapter is the final model.')
